In [ ]:
import ast

tree = ast.parse(open("nlp_06-Copy1.py").read())

print("Classes:")
for n in ast.walk(tree):
    if isinstance(n, ast.ClassDef):
        print("  ", n.name)

print("\nFunctions:")
for n in ast.walk(tree):
    if isinstance(n, ast.FunctionDef):
        print("  ", n.name)


In [12]:

import ast
from collections import defaultdict, Counter
from pathlib import Path

# -----------------------------
# Datencontainer
# -----------------------------

class DependencyStats:
    def __init__(self):
        self.calls = defaultdict(Counter)
        self.type_uses = defaultdict(Counter)
        self.runtime_uses = defaultdict(Counter)

# -----------------------------
# AST Visitor
# -----------------------------

class DependencyVisitor(ast.NodeVisitor):
    def __init__(self):
        self.current = None
        self.stats = DependencyStats()

    # ---- Context tracking ----

    def visit_ClassDef(self, node):
        prev = self.current
        self.current = node.name
        self.generic_visit(node)
        self.current = prev

    def visit_FunctionDef(self, node):
        prev = self.current
        self.current = node.name

        if node.returns:
            self._handle_annotation(node.returns)

        for arg in node.args.args:
            if arg.annotation:
                self._handle_annotation(arg.annotation)

        self.generic_visit(node)
        self.current = prev

    # ---- Runtime calls ----

    def visit_Call(self, node):
        if self.current:
            if isinstance(node.func, ast.Name):
                self.stats.calls[self.current][node.func.id] += 1
            elif isinstance(node.func, ast.Attribute):
                self.stats.calls[self.current][node.func.attr] += 1
        self.generic_visit(node)

    # ---- Runtime attribute usage ----

    def visit_Attribute(self, node):
        if self.current and isinstance(node.value, ast.Name):
            self.stats.runtime_uses[self.current][node.value.id] += 1
        self.generic_visit(node)

    # ---- Type annotations ----

    def visit_AnnAssign(self, node):
        self._handle_annotation(node.annotation)
        self.generic_visit(node)

    # ---- Helpers ----

    def _handle_annotation(self, node):
        if not self.current:
            return
        for name in self._extract_names(node):
            self.stats.type_uses[self.current][name] += 1

    def _extract_names(self, node):
        names = []
        if isinstance(node, ast.Name):
            names.append(node.id)
        elif isinstance(node, ast.Attribute):
            names.append(node.attr)
        elif isinstance(node, ast.Subscript):
            names += self._extract_names(node.value)
            names += self._extract_names(node.slice)
        elif isinstance(node, ast.Tuple):
            for e in node.elts:
                names += self._extract_names(e)
        return names

# -----------------------------
# Analyse
# -----------------------------

def analyze_file(path: str) -> DependencyStats:
    tree = ast.parse(Path(path).read_text())
    visitor = DependencyVisitor()
    visitor.visit(tree)
    return visitor.stats

def print_summary(stats: DependencyStats):
    print("\n=== CALL COUPLING ===")
    for src, targets in stats.calls.items():
        print(f"{src} -> {dict(targets)}")

    print("\n=== TYPE-ONLY COUPLING ===")
    for src, targets in stats.type_uses.items():
        print(f"{src} -> {dict(targets)}")

    print("\n=== RUNTIME ATTRIBUTE COUPLING ===")
    for src, targets in stats.runtime_uses.items():
        print(f"{src} -> {dict(targets)}")

def import_risk_report(stats: DependencyStats):
    print("\n=== IMPORT RISK REPORT ===")
    for entity in set(stats.type_uses) | set(stats.runtime_uses):
        type_only = sum(stats.type_uses[entity].values())
        runtime = sum(stats.runtime_uses[entity].values())
        if type_only > 0 and runtime == 0:
            print(
                f"{entity}: type-only dependency ({type_only}) "
                f"→ candidate for domain/types module"
            )

# -----------------------------
# Usage
# -----------------------------

if __name__ == "__main__":
    stats = analyze_file("nlp_06-Copy1.py")
    print_summary(stats)
    import_risk_report(stats)



=== CALL COUPLING ===
__init__ -> {'_validate_class_contract': 1, '_validate_config': 1, '__init__': 4, 'super': 4, 'LlmConfig': 1, 'get': 1, 'LocalLlamaLLM': 3, 'ValueError': 2, 'build_prompt': 1, '_validate_specs': 1, 'fields': 1, 'warning': 1, 'Path': 2, 'load_nodes_from_yaml': 1, 'configure_logger': 2, 'PipelineOrchestrator': 1, 'str': 2}
_validate_class_contract -> {'_is_loader': 1, 'TypeError': 3}
_validate_config -> {'isinstance': 1, 'TypeError': 1, 'type': 1}
get -> {'KeyError': 1}
load -> {'_read_file': 1, '_parse_yaml': 1, '_postprocess': 1, 'open': 2, 'load': 3, 'obj_type': 1, 'Path': 1, 'replace': 1, 'exists': 1, 'FileNotFoundError': 1, 'get': 1, 'serializer_cls': 1, 'read_bytes': 1, 'hexdigest': 1, 'sha256': 1, 'ValueError': 1}
_read_file -> {'Path': 1, 'exists': 1, 'FileNotFoundError': 1, 'read_text': 1}
_parse_yaml -> {'safe_load': 1, 'ValueError': 1}
ComponentSpec -> {'dataclass': 1}
_call -> {'NotImplementedError': 1, 'post': 1, 'raise_for_status': 1, 'json': 1, 'get'

In [13]:
import ast
import os
from typing import Dict, List, Set, Tuple, Optional
from collections import defaultdict

# ==============================================
# AST-Basierte Code-Analyse
# ==============================================

class CodeAnalyzer(ast.NodeVisitor):
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.classes: Dict[str, Dict] = {}
        self.functions: Dict[str, Dict] = {}
        self.imports: Dict[str, Set[str]] = defaultdict(set)
        self.calls: Dict[str, Set[str]] = defaultdict(set)
        self.current_class: Optional[str] = None
        self.current_function: Optional[str] = None

        self._parse_file()

    def _parse_file(self):
        with open(self.file_path, "r", encoding="utf-8") as f:
            tree = ast.parse(f.read(), filename=self.file_path)
            self.visit(tree)

    # -------------------------
    # Klassen- und Methodenbesuche
    # -------------------------
    def visit_ClassDef(self, node: ast.ClassDef):
        self.current_class = node.name
        self.classes[node.name] = {
            "methods": [],
            "bases": [b.id if isinstance(b, ast.Name) else ast.dump(b) for b in node.bases],
            "docstring": ast.get_docstring(node),
        }
        self.generic_visit(node)
        self.current_class = None

    def visit_FunctionDef(self, node: ast.FunctionDef):
        func_name = node.name
        if self.current_class:
            self.classes[self.current_class]["methods"].append(func_name)
            full_name = f"{self.current_class}.{func_name}"
        else:
            self.functions[func_name] = {"docstring": ast.get_docstring(node)}
            full_name = func_name

        self.current_function = full_name
        self.generic_visit(node)
        self.current_function = None

    # -------------------------
    # Funktionsaufrufe sammeln
    # -------------------------
    def visit_Call(self, node: ast.Call):
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
        elif isinstance(node.func, ast.Attribute):
            func_name = node.func.attr
        else:
            func_name = ast.dump(node.func)

        if self.current_function:
            self.calls[self.current_function].add(func_name)
        self.generic_visit(node)

    # -------------------------
    # Imports sammeln
    # -------------------------
    def visit_Import(self, node: ast.Import):
        for alias in node.names:
            self.imports[self.file_path].add(alias.name)
        self.generic_visit(node)

    def visit_ImportFrom(self, node: ast.ImportFrom):
        module = node.module or ""
        for alias in node.names:
            self.imports[self.file_path].add(f"{module}.{alias.name}")
        self.generic_visit(node)

# ==============================================
# Automatische Cluster-Erstellung
# ==============================================
def cluster_by_call_graph(calls: Dict[str, Set[str]]) -> List[Set[str]]:
    """Einfache Cluster-Logik: Funktionen, die sich gegenseitig aufrufen, werden zusammengefasst."""
    clusters: List[Set[str]] = []
    visited: Set[str] = set()

    for func in calls:
        if func in visited:
            continue
        cluster = set()
        stack = [func]
        while stack:
            f = stack.pop()
            if f not in cluster:
                cluster.add(f)
                stack.extend(calls.get(f, []))
        visited.update(cluster)
        clusters.append(cluster)
    return clusters

# ==============================================
# Checkliste für Modulzuordnung
# ==============================================
def can_class_go_to_module(class_name: str, module_name: str, rules: Dict[str, Set[str]]) -> bool:
    """
    Regeln: 
      rules = {
          "module_name": {"allowed_class1", "allowed_class2"},
      }
    """
    allowed = rules.get(module_name, set())
    return class_name in allowed

# ==============================================
# Beispiel für Nutzung
# ==============================================
if __name__ == "__main__":
    # Alle Python-Dateien im Projekt analysieren
    project_dir = "./src"
    analyzer = None
    all_calls = defaultdict(set)
    all_classes = {}
    all_imports = defaultdict(set)

    for root, _, files in os.walk(project_dir):
        for f in files:
            if f.endswith(".py"):
                path = os.path.join(root, f)
                analyzer = CodeAnalyzer(path)
                all_calls.update(analyzer.calls)
                all_classes.update(analyzer.classes)
                all_imports.update(analyzer.imports)

    # Clustering
    clusters = cluster_by_call_graph(all_calls)
    print("Automatische Funktions-/Methoden-Cluster:")
    for i, c in enumerate(clusters):
        print(f"Cluster {i+1}: {c}")

    # Checkliste Beispiel
    rules = {
        "llm_module": {"LocalLlamaLLM", "HeadPromptLlmConfig"},
        "pipeline_module": {"PipelineOrchestrator", "BaseComponent"},
    }
    for cls in all_classes:
        for mod in rules:
            print(f"Darf {cls} in {mod}? {can_class_go_to_module(cls, mod, rules)}")


Automatische Funktions-/Methoden-Cluster:
